# Stock Price Data Exploration

This notebook loads and explores historical stock price data for major tech companies.

**Tickers covered:** AAPL, MSFT, GOOGL, AMZN, TSLA, NVDA, META

## 1. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries loaded successfully!")

Libraries loaded successfully!


## 2. Load Stock Price Data

In [2]:
DATA_DIR = Path("../data/raw/prices")
TICKERS = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA", "NVDA", "META"]

def load_stock_csv(file_path):
    """Load stock CSV handling standard and yfinance multi-header formats."""
    first_row = pd.read_csv(file_path, nrows=0)
    if 'Date' in first_row.columns:
        df = pd.read_csv(file_path)
        df['Date'] = pd.to_datetime(df['Date'])
    elif 'Price' in first_row.columns:
        df = pd.read_csv(file_path, header=[0, 1], index_col=0)
        df = df.dropna(how='all')
        df.index = pd.to_datetime(df.index)
        df.index.name = 'Date'
        df.columns = df.columns.droplevel(1)
        df = df.reset_index()
        for col in ['Close', 'High', 'Low', 'Open', 'Volume']:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
    else:
        raise ValueError(f"Unknown CSV format in {file_path}")
    return df

stock_data = {}
for ticker in TICKERS:
    file_path = DATA_DIR / f"{ticker}.csv"
    if file_path.exists():
        try:
            df = load_stock_csv(file_path)
            stock_data[ticker] = df
            print(f"{ticker}: {len(df)} rows ({df['Date'].min().date()} to {df['Date'].max().date()})")
        except Exception as e:
            print(f"{ticker}: Error - {e}")
    else:
        print(f"{ticker}: File not found")

print(f"\nTotal tickers loaded: {len(stock_data)}")

AAPL: 251 rows (2024-01-02 to 2024-12-30)
MSFT: 1253 rows (2021-07-30 to 2026-07-28)
GOOGL: 1253 rows (2021-07-30 to 2026-07-28)
AMZN: 1253 rows (2021-07-30 to 2026-07-28)
TSLA: 1253 rows (2021-07-30 to 2026-07-28)
NVDA: 1253 rows (2021-07-30 to 2026-07-28)
META: 1253 rows (2021-07-30 to 2026-07-28)

Total tickers loaded: 7


## 3. Inspect Data Structure

In [3]:
sample_ticker = TICKERS[0]
print(f"Sample data for {sample_ticker}:")
print(f"Shape: {stock_data[sample_ticker].shape}")
print(f"\nColumns: {list(stock_data[sample_ticker].columns)}")
print(f"\nData types:")
print(stock_data[sample_ticker].dtypes)
print(f"\nFirst 5 rows:")
stock_data[sample_ticker].head()

Sample data for AAPL:
Shape: (251, 6)

Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']

Data types:
Date      datetime64[us]
Open             float64
High             float64
Low              float64
Close            float64
Volume             int64
dtype: object

First 5 rows:


,Date,Open,High,Low,Close,Volume
0,2024-01-02,185.06,186.33,181.83,183.56,82488700
1,2024-01-03,182.16,183.80,181.38,182.19,58414500
2,2024-01-04,180.11,181.04,178.86,179.87,71983600
3,2024-01-05,179.95,180.71,178.15,179.15,62379700
4,2024-01-08,180.05,183.52,179.47,183.48,59144500


In [4]:
print(f"Last 5 rows for {sample_ticker}:")
stock_data[sample_ticker].tail()

Last 5 rows for AAPL:


,Date,Open,High,Low,Close,Volume
246,2024-12-23,253.15,254.03,251.84,253.65,40858800
247,2024-12-24,253.87,256.57,253.67,256.56,23234700
248,2024-12-26,256.55,258.45,255.99,257.38,27237100
249,2024-12-27,256.19,257.06,251.45,253.97,42355300
250,2024-12-30,250.63,251.89,249.16,250.60,35557500


## 4. Data Summary & Statistics

In [5]:
for ticker, df in stock_data.items():
    print(f"\n{'='*60}")
    print(f"  {ticker} Summary")
    print(f"{'='*60}")
    print(f"Date Range: {df['Date'].min().date()} to {df['Date'].max().date()}")
    print(f"Total Trading Days: {len(df)}")
    print(f"Missing Values: {df.isnull().sum().sum()}")
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if 'Close' in numeric_cols:
        print(f"Close - Min: ${df['Close'].min():.2f}, Max: ${df['Close'].max():.2f}, Mean: ${df['Close'].mean():.2f}")
    if 'Volume' in numeric_cols:
        print(f"Avg Daily Volume: {df['Volume'].mean():,.0f}")


  AAPL Summary
Date Range: 2024-01-02 to 2024-12-30
Total Trading Days: 251
Missing Values: 0
Close - Min: $163.36, Max: $257.38, Mean: $205.28
Avg Daily Volume: 57,177,003

  MSFT Summary
Date Range: 2021-07-30 to 2026-07-28
Total Trading Days: 1253
Missing Values: 0
Close - Min: $207.73, Max: $538.66, Mean: $360.59
Avg Daily Volume: 26,630,845

  GOOGL Summary
Date Range: 2021-07-30 to 2026-07-28
Total Trading Days: 1253
Missing Values: 0
Close - Min: $82.70, Max: $402.38, Mean: $171.47
Avg Daily Volume: 32,345,782

  AMZN Summary
Date Range: 2021-07-30 to 2026-07-28
Total Trading Days: 1253
Missing Values: 0
Close - Min: $81.82, Max: $274.99, Mean: $171.53
Avg Daily Volume: 55,075,480

  TSLA Summary
Date Range: 2021-07-30 to 2026-07-28
Total Trading Days: 1253
Missing Values: 0
Close - Min: $108.10, Max: $489.88, Mean: $284.81
Avg Daily Volume: 95,771,803

  NVDA Summary
Date Range: 2021-07-30 to 2026-07-28
Total Trading Days: 1253
Missing Values: 0
Close - Min: $11.20, Max: $235.

In [6]:
print(f"Detailed Statistics for {sample_ticker}:")
stock_data[sample_ticker].describe()

Detailed Statistics for AAPL:


,Date,Open,High,Low,Close,Volume
count,251,251.00,251.00,251.00,251.00,251.00
mean,2024-07-01 02:11:57.131474,205.02,206.97,203.31,205.28,57177003.19
min,2024-01-02 00:00:00,163.71,164.75,162.45,163.36,23234700.00
25%,2024-04-02 12:00:00,181.79,183.35,180.49,182.25,41871250.00
50%,2024-07-02 00:00:00,212.09,214.92,210.10,212.40,49889100.00
75%,2024-09-30 12:00:00,225.69,227.54,223.63,225.65,62958150.00
max,2024-12-30 00:00:00,256.55,258.45,255.99,257.38,318679900.00
std,NaN,25.27,25.53,25.08,25.46,30764411.81


## 5. Check Data Quality

In [7]:
print("Missing Values Summary:")
print(f"{'Ticker':<10} {'Total Missing':<15} {'% Missing':<10}")
print("-" * 35)
for ticker, df in stock_data.items():
    total_missing = df.isnull().sum().sum()
    total_cells = df.shape[0] * df.shape[1]
    pct_missing = (total_missing / total_cells) * 100
    print(f"{ticker:<10} {total_missing:<15} {pct_missing:.2f}%")

Missing Values Summary:
Ticker     Total Missing   % Missing 
-----------------------------------
AAPL       0               0.00%
MSFT       0               0.00%
GOOGL      0               0.00%
AMZN       0               0.00%
TSLA       0               0.00%
NVDA       0               0.00%
META       0               0.00%


In [8]:
print("Duplicate Dates Check:")
for ticker, df in stock_data.items():
    duplicates = df['Date'].duplicated().sum()
    print(f"{ticker}: {duplicates} duplicate dates")

Duplicate Dates Check:
AAPL: 0 duplicate dates
MSFT: 0 duplicate dates
GOOGL: 0 duplicate dates
AMZN: 0 duplicate dates
TSLA: 0 duplicate dates
NVDA: 0 duplicate dates
META: 0 duplicate dates


## 6. Combine All Stocks (Close Prices)

In [9]:
close_prices = pd.DataFrame()
for ticker, df in stock_data.items():
    if 'Close' in df.columns:
        temp = df[['Date', 'Close']].copy()
        temp = temp.rename(columns={'Close': ticker}).set_index('Date')
        if close_prices.empty:
            close_prices = temp
        else:
            close_prices = close_prices.join(temp, how='outer')

close_prices = close_prices.sort_index()
print(f"Combined Close Prices - Shape: {close_prices.shape}")
print(f"Date Range: {close_prices.index.min().date()} to {close_prices.index.max().date()}")
close_prices.head()

Combined Close Prices - Shape: (1253, 7)
Date Range: 2021-07-30 to 2026-07-28


,AAPL,MSFT,GOOGL,AMZN,TSLA,NVDA,META
Date,,,,,,,
2021-07-30,NaN,273.43,133.54,166.38,229.07,19.43,353.20
2021-08-02,NaN,273.34,133.67,166.57,236.56,19.68,348.89
2021-08-03,NaN,275.55,134.44,168.31,236.58,19.75,348.18
2021-08-04,NaN,274.96,133.94,167.74,236.97,20.20,355.80
2021-08-05,NaN,277.85,135.05,168.80,238.21,20.57,359.81


In [10]:
print("Correlation Matrix (Close Prices):")
close_prices.corr()

Correlation Matrix (Close Prices):


,AAPL,MSFT,GOOGL,AMZN,TSLA,NVDA,META
AAPL,1.00,0.55,0.68,0.65,0.77,0.82,0.70
MSFT,0.55,1.00,0.65,0.85,0.51,0.87,0.94
GOOGL,0.68,0.65,1.00,0.85,0.74,0.89,0.74
AMZN,0.65,0.85,0.85,1.00,0.74,0.90,0.92
TSLA,0.77,0.51,0.74,0.74,1.00,0.64,0.60
NVDA,0.82,0.87,0.89,0.90,0.64,1.00,0.93
META,0.70,0.94,0.74,0.92,0.60,0.93,1.00


## 7. Performance Comparison

In [11]:
print("Stock Performance:")
for ticker in close_prices.columns:
    col = close_prices[ticker].dropna()
    if len(col) > 0:
        change = ((col.iloc[-1] / col.iloc[0]) - 1) * 100
        symbol = "+" if change > 0 else ""
        print(f"  {ticker}: {symbol}{change:.1f}%")

Stock Performance:
  AAPL: +36.5%
  MSFT: +43.9%
  GOOGL: +149.9%
  AMZN: +38.8%
  TSLA: +34.2%
  NVDA: +913.9%
  META: +68.0%


## 8. Daily Returns & Volatility

In [12]:
daily_returns = close_prices.pct_change().dropna(how='all')
print("Daily Returns Summary:")
print(daily_returns.describe())

Daily Returns Summary:
        AAPL    MSFT   GOOGL    AMZN    TSLA    NVDA    META
count 250.00 1252.00 1252.00 1252.00 1252.00 1252.00 1252.00
mean    0.00    0.00    0.00    0.00    0.00    0.00    0.00
std     0.01    0.02    0.02    0.02    0.04    0.03    0.03
min    -0.05   -0.10   -0.10   -0.14   -0.15   -0.17   -0.26
25%    -0.01   -0.01   -0.01   -0.01   -0.02   -0.02   -0.01
50%     0.00    0.00    0.00    0.00    0.00    0.00    0.00
75%     0.01    0.01    0.01    0.01    0.02    0.02    0.01
max     0.07    0.10    0.10    0.14    0.23    0.24    0.23


In [13]:
print("\nAnnualized Volatility (252 trading days):")
volatility = daily_returns.std() * np.sqrt(252)
for ticker, vol in volatility.sort_values(ascending=False).items():
    print(f"  {ticker}: {vol:.2%}")


Annualized Volatility (252 trading days):
  TSLA: 59.69%
  NVDA: 51.88%
  META: 44.56%
  AMZN: 35.61%
  GOOGL: 31.82%
  MSFT: 27.13%
  AAPL: 22.45%


---
## Summary

Key observations:
- Data loaded from CSV files in `data/raw/prices/`
- Handled both standard and yfinance multi-header CSV formats
- Combined close prices DataFrame created for cross-stock analysis
- Daily returns and volatility computed

**Next Steps:**
- Feature engineering (moving averages, RSI, MACD)
- Merge with news sentiment data
- Model development